# Stratification timing: runtime vs. dimension $d$

Runtime of **tensor stratification** on the SphereLab problem, as a function of the axis
dimension $d$ of a cubical $d \times d \times d$ tensor with a hidden sphere.

The input is `labs/SphereLab.ipynb` section 3's construction at scale: axis values
$u_i = x_i^2 - r^2/3$ with the support where the three $u$'s sum to zero, so the sphere's
equation holds exactly on the lattice; then a random **orthogonal** change of basis on every
axis, and `nondeg`. That is what hides the sphere.

`stratify` is two phases, and they are timed apart because only the first depends on the
method:

| column | phase |
|---|---|
| `der_seconds` | **solve the derivation** — `derTrOpsReduced`, the null-space problem |
| `strat_seconds` | **compute the stratification** — real canonical form per axis, then act on $\Gamma$ |

**The grid.** $d = 10, 15, 20, \ldots$; `Float32` solved to `tol = 1e-8` and `Float64` to
`tol = 1e-16`; the universal chisel against two operator spaces, `universal`
(`UniversalOp()`, unrestricted) and `symmetric` (`SymmetricOp()`); every applicable
derivation method.

**Drop-out.** A method whose total time passes **60 s** at some $d$ is not run at any larger
$d$. Its curve simply ends, and where it ends is the result — not missing data.

In [ ]:
using Pkg
Pkg.activate("..")          # the OpenDleto project

using CSV, DataFrames, PlotlyJS

PlotlyJS.templates.default = "plotly_white"

df = CSV.read("stratify-timing.csv", DataFrame)
println(nrow(df), " rows, d = ", minimum(df.d), ":", maximum(df.d))
first(df, 5)

In [ ]:
# A run that errored carries no time, so the plots below use `ok` only.  The usual causes
# are a method that does not apply in that operator space and a tolerance under machine
# epsilon that leaves no mode to call null -- both worth seeing, so they are printed.

ok = filter(:status => ==("ok"), df)
println(nrow(df) - nrow(ok), " of ", nrow(df), " runs did not complete")

bad = filter(:status => !=("ok"), df)
if nrow(bad) > 0
    display(sort(combine(groupby(bad, [:method, :ops, :status]), nrow => :runs), :runs, rev=true))
end

In [ ]:
# The palette.  Colour follows the METHOD, so a method keeps its colour in every figure
# even where it is absent from a panel.  These are the first six slots of the validated
# categorical order (worst adjacent CVD ΔE 9.1, normal-vision ΔE 19.6 on the light surface);
# three of them sit under 3:1 contrast, which is why the tables below are not optional.

const METHOD_ORDER = ["SylverLining/Auto", "SylverLining/SVD", "Auto",
                      "QuickDer", "QuickDer3", "SymmetricGram"]

const METHOD_COLOR = Dict(
    "SylverLining/Auto" => "#2a78d6",   # blue
    "SylverLining/SVD"  => "#eb6834",   # orange
    "Auto"              => "#1baf7a",   # aqua
    "QuickDer"          => "#eda100",   # yellow
    "QuickDer3"         => "#e87ba4",   # magenta
    "SymmetricGram"     => "#008300",   # green
)

const INK     = "#52514e"     # text ink: labels never wear the series colour
const SURFACE = "#fcfcfb"

# The 2x2 facet: precision down the rows, operator space across the columns.
const PANELS = [("Float32", "universal", 1, 1), ("Float32", "symmetric", 1, 2),
                ("Float64", "universal", 2, 1), ("Float64", "symmetric", 2, 2)]

const PANEL_TITLES = ["Float32 · tol 1e-8 · universal ops"   "Float32 · tol 1e-8 · symmetric ops"
                      "Float64 · tol 1e-16 · universal ops"  "Float64 · tol 1e-16 · symmetric ops"]

panel_rows(data, elt, o) = data[(data.eltype .== elt) .& (data.ops .== o), :]
series(data, m) = sort(data[data.method .== m, :], :d)
nothing

## The table

The numbers first: total seconds by $d$ against method, one table per precision and
operator space. A `missing` is a method that had already passed the 60 s budget at a
smaller $d$, or one that does not apply in that operator space.

In [ ]:
for (elt, o, _, _) in PANELS
    sub = panel_rows(ok, elt, o)
    isempty(sub) && continue
    piv = sort(unstack(sub, :d, :method, :total_seconds), :d)
    # Keep the method columns in the canonical order.
    cols = [:d; Symbol.(filter(m -> string(m) in names(piv), METHOD_ORDER))]
    println("\ntotal seconds — ", elt, " / ", o, " operators")
    display(piv[!, cols])
end

## Total runtime

Log $y$: each method is a straight line only if its cost is a clean power of $d$. Where a
curve **stops** is where that method passed the 60 s budget — the drop-out points are the
headline.

In [ ]:
"""
    facet_figure(data, ycol; title, ylab, logy = true, endlabels = false)

One 2x2 figure: precision down the rows, operator space across the columns, one line per
method.  `endlabels` writes the method name at the last measured point, which is where the
drop-out happens and therefore where the labels do not collide.
"""
function facet_figure(data, ycol; title, ylab, logy = true, endlabels = false)
    p = make_subplots(rows = 2, cols = 2, subplot_titles = PANEL_TITLES,
                      shared_xaxes = true, shared_yaxes = true,
                      vertical_spacing = 0.13, horizontal_spacing = 0.06)
    seen = Set{String}()
    for (elt, o, r, c) in PANELS
        panel = panel_rows(data, elt, o)
        for m in METHOD_ORDER
            s = series(panel, m)
            nrow(s) == 0 && continue
            show_it = !(m in seen)
            push!(seen, m)
            labels = [endlabels && i == nrow(s) ? " " * m : "" for i in 1:nrow(s)]
            add_trace!(p, scatter(
                x = s.d, y = s[!, ycol],
                mode = endlabels ? "lines+markers+text" : "lines+markers",
                name = m, legendgroup = m, showlegend = show_it,
                line = attr(color = METHOD_COLOR[m], width = 2),
                marker = attr(color = METHOD_COLOR[m], size = 8,
                              line = attr(color = SURFACE, width = 2)),
                text = labels, textposition = "middle right",
                textfont = attr(color = INK, size = 11),
                hovertemplate = "d = %{x}<br>%{y:.4g}<extra>" * m * "</extra>",
            ), row = r, col = c)
        end
    end
    relayout!(p, title_text = title, height = 760, width = 1000, hovermode = "x unified",
              legend = attr(orientation = "h", y = -0.09, x = 0),
              margin = attr(l = 80, r = 130, t = 100, b = 110))
    if logy
        relayout!(p, yaxis_type = "log", yaxis2_type = "log",
                     yaxis3_type = "log", yaxis4_type = "log")
    end
    relayout!(p, xaxis3_title_text = "axis dimension d", xaxis4_title_text = "axis dimension d",
                 yaxis_title_text = ylab, yaxis3_title_text = ylab)
    return p
end

facet_figure(ok, :total_seconds;
             title = "Stratification: total runtime vs. dimension (60 s budget, log scale)",
             ylab = "seconds", endlabels = true)

## Where the time goes

Solid is the **derivation solve**, dotted is the **stratification** that follows it, on one
shared log axis so the gap between them is the real ratio. The stratification is an
eigendecomposition per axis plus three contractions — it does not depend on the method,
which is why the dotted lines lie on top of one another within a panel.

In [ ]:
p = make_subplots(rows = 2, cols = 2, subplot_titles = PANEL_TITLES,
                  shared_xaxes = true, shared_yaxes = true,
                  vertical_spacing = 0.13, horizontal_spacing = 0.06)
seen = Set{String}()
for (elt, o, r, c) in PANELS
    panel = panel_rows(ok, elt, o)
    for m in METHOD_ORDER
        s = series(panel, m)
        nrow(s) == 0 && continue
        show_it = !(m in seen)
        push!(seen, m)
        add_trace!(p, scatter(
            x = s.d, y = s.der_seconds, mode = "lines+markers",
            name = m, legendgroup = m, showlegend = show_it,
            line = attr(color = METHOD_COLOR[m], width = 2),
            marker = attr(color = METHOD_COLOR[m], size = 8,
                          line = attr(color = SURFACE, width = 2)),
            hovertemplate = "d = %{x}<br>derivation %{y:.4g} s<extra>" * m * "</extra>",
        ), row = r, col = c)
        add_trace!(p, scatter(
            x = s.d, y = s.strat_seconds, mode = "lines",
            name = m, legendgroup = m, showlegend = false,
            line = attr(color = METHOD_COLOR[m], width = 2, dash = "dot"),
            hovertemplate = "d = %{x}<br>stratification %{y:.4g} s<extra>" * m * "</extra>",
        ), row = r, col = c)
    end
end

# A key for the dash encoding, in ink rather than in any series colour.
add_trace!(p, scatter(x = Float64[], y = Float64[], mode = "lines", name = "solid: derivation solve",
                      line = attr(color = INK, width = 2)), row = 1, col = 1)
add_trace!(p, scatter(x = Float64[], y = Float64[], mode = "lines", name = "dotted: stratification",
                      line = attr(color = INK, width = 2, dash = "dot")), row = 1, col = 1)

relayout!(p, title_text = "Derivation solve vs. the stratification that follows it (log scale)",
          height = 760, width = 1000, hovermode = "closest",
          legend = attr(orientation = "h", y = -0.09, x = 0),
          margin = attr(l = 80, r = 40, t = 100, b = 110),
          yaxis_type = "log", yaxis2_type = "log", yaxis3_type = "log", yaxis4_type = "log",
          xaxis3_title_text = "axis dimension d", xaxis4_title_text = "axis dimension d",
          yaxis_title_text = "seconds", yaxis3_title_text = "seconds")
p

### The same split as a proportion

`der_seconds / total_seconds`. At 1.0 the derivation solve is the entire cost and the
stratification is free.

In [ ]:
ok.der_frac = ok.der_seconds ./ ok.total_seconds

facet_figure(ok, :der_frac;
             title = "Fraction of total time spent solving for the derivation",
             ylab = "der / total", logy = false)

## Did it actually find the sphere?

`lsq_err` is the relative reconstruction error inside the permutation-and-scale ambiguity a
stratification is defined up to: 0 is perfect, ~1 is nothing recovered. A timing table is
only worth reading if the runs it times solved the problem.

In [ ]:
facet_figure(ok, :lsq_err;
             title = "Recovery of the hidden sphere (log scale; 0 perfect, ~1 nothing)",
             ylab = "relative reconstruction error")

**Reading that figure.** The two operator spaces are not two solvers of the same problem.

The **symmetric** space is the well-posed one here: an orthogonal conjugate of a diagonal
derivation is symmetric, so `SymmetricOp()` is exactly the space that fits the scramble.
There the derivation space has nullity 3 — the two scalar derivations plus the sphere's —
and recovery lands around `1e-6` in `Float32` and `1e-14` in `Float64`, which is the
tolerance doing its job.

The **universal** space finds a much larger derivation space on this input (nullity 13–14
already at $d = 10$), so a random combination of its basis does not single out the sphere
and `lsq_err` sits near 1. That is a property of the input under an unrestricted chisel, not
a solver failure. The universal panels are here for the runtime an unrestricted chisel
costs; their `lsq_err` should be read as *this chisel does not pin the sphere down*.

Both tolerances sit just under the respective machine epsilon (`eps(Float32) = 1.2e-7`,
`eps(Float64) = 2.2e-16`), which is deliberate: `tol` is the singular-value cutoff that
separates the null space from the rest, so this asks each solver to call a mode null only
when it is null to the last bit. The `nullity` column confirms both settings still find the
three-dimensional derivation space under symmetric operators.

## Reproducing this

From the repository root, with the project's Julia wrapper — `bench/jl` holds the thread,
heap and RSS budget for a shared machine, so never invoke bare `julia` here:

```bash
bench/jl timing/StratifyTiming.jl 200 60      # maxd 200, 60 s budget
```

The driver is [`StratifyTiming.jl`](StratifyTiming.jl); it runs `stratify(Ω, ch, Γ)`'s own
body unrolled so the two timed phases are the real ones. Input construction and the
reconstruction score come from `bench/SphereHarness.jl`. Delete
`timing/stratify-timing.csv` first for a clean sweep — the driver appends.